
1. TRAFFIC VEHICLE COUNTING
2. YOLO26m + BoT-SORT
3. Track-level class
4. Bottom-center trajectory
5. Diagonal virtual counting line
6. Direction detection
7. Motorcycle fragmentation deduplication
8. CFR FFmpeg output

# PHASE 1

OUTPUT:
tracks_raw.csv

IMPORTANT:
raw class_name is preserved ONLY as input to Phase 2.
Phase 3 will NEVER use raw class_name.

## Configuration

In [1]:
!pip install -q -U ultralytics opencv-python-headless psutil

In [2]:
from pathlib import Path
from collections import defaultdict
import json
import math
import os
import subprocess
import time
import warnings

import cv2
import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO

warnings.filterwarnings("ignore")


# ============================================================
# PATHS
# ============================================================

VIDEO_PATH = Path(
    "/kaggle/input/datasets/chrisbiran/traffic-tracker-videos/TOR3-PAGI.mp4"
)

PROJECT_DIR = Path(
    "/kaggle/working/traffic_counting"
)

PHASE1_DIR = PROJECT_DIR / "phase1"
PHASE2_DIR = PROJECT_DIR / "phase2"
PHASE3_DIR = PROJECT_DIR / "phase3"

for directory in [
    PHASE1_DIR,
    PHASE2_DIR,
    PHASE3_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# MODEL
# ============================================================

MODEL_NAME = "yolo26m.pt"

TRACKER = "botsort.yaml"

DEVICE = 0 if torch.cuda.is_available() else "cpu"


# ============================================================
# DETECTION CONFIG
# ============================================================

IMG_SIZE = 960

CONF_THRESHOLD = 0.25

IOU_THRESHOLD = 0.50


# ============================================================
# TARGET CLASSES
# ============================================================

TARGET_CLASSES = {
    "person",
    "motorcycle",
    "car",
    "truck",
    "bus",
}


# ============================================================
# VEHICLE CLASSES
# ============================================================

VEHICLE_CLASSES = {
    "motorcycle",
    "car",
    "truck",
    "bus",
}


# ============================================================
# DIAGONAL COUNTING LINE
#
# Upper point
# Lower point
# ============================================================

COUNT_LINE = {
    "orientation": "diagonal",

    "x1": None,
    "y1": None,

    "x2": None,
    "y2": None,
}


# ============================================================
# COUNTING PARAMETERS
# ============================================================

# Distance from line where a point is considered "near line".
# This prevents tiny jitter around the line from creating
# false crossing events.
LINE_DEADBAND_PX = 8.0


# Minimum number of trajectory observations required.
MIN_TRACK_OBSERVATIONS = 5


# Maximum allowed time gap between consecutive observations
# when analyzing trajectory.
MAX_TRAJECTORY_GAP_SEC = 1.5


# ============================================================
# MOTORCYCLE FRAGMENTATION DEDUP
#
# Conservative on purpose.
# We only suppress motorcycle events when BOTH:
#   1. crossing events are close in time
#   2. crossing points are close in space
#   3. direction is identical
#
# This is NOT aggressive duplicate removal.
# ============================================================

MOTO_DEDUP_TIME_SEC = 1.50

MOTO_DEDUP_DISTANCE_PX = 80.0


# ============================================================
# OUTPUTS
# ============================================================

PHASE1_TRACKS_PATH = (
    PHASE1_DIR /
    "tracks_raw.csv"
)

PHASE2_TRACKS_PATH = (
    PHASE2_DIR /
    "tracks_with_track_class.csv"
)

PHASE2_SUMMARY_PATH = (
    PHASE2_DIR /
    "track_quality_summary.csv"
)

PHASE3_CROSSINGS_PATH = (
    PHASE3_DIR /
    "crossing_events.csv"
)

PHASE3_FINAL_COUNTS_PATH = (
    PHASE3_DIR /
    "final_vehicle_counts.csv"
)

PHASE3_TRACKED_VIDEO_PATH = (
    PHASE3_DIR /
    "tracked_video_diagonal_counting_CFR.mp4"
)


print("=" * 70)
print("PROJECT CONFIGURATION")
print("=" * 70)

print("Video       :", VIDEO_PATH)
print("Model       :", MODEL_NAME)
print("Tracker     :", TRACKER)
print("Device      :", DEVICE)
print("Image size  :", IMG_SIZE)
print("Confidence  :", CONF_THRESHOLD)

PROJECT CONFIGURATION
Video       : /kaggle/input/datasets/chrisbiran/traffic-tracker-videos/TOR3-PAGI.mp4
Model       : yolo26m.pt
Tracker     : botsort.yaml
Device      : 0
Image size  : 960
Confidence  : 0.25


Check environment

In [3]:
# ============================================================
# ENVIRONMENT CHECK
# ============================================================

import ultralytics

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("Python          :", os.sys.version.split()[0])
print("PyTorch         :", torch.__version__)
print("Ultralytics    :", ultralytics.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("CUDA version    :", torch.version.cuda)

if torch.cuda.is_available():

    print(
        "GPU count       :",
        torch.cuda.device_count()
    )

    for i in range(torch.cuda.device_count()):

        print(
            f"GPU {i}          :",
            torch.cuda.get_device_name(i)
        )

ENVIRONMENT
Python          : 3.12.13
PyTorch         : 2.10.0+cu128
Ultralytics    : 8.4.120
CUDA available  : True
CUDA version    : 12.8
GPU count       : 2
GPU 0          : Tesla T4
GPU 1          : Tesla T4


Read video metadata

In [4]:
# ============================================================
# VIDEO METADATA
# ============================================================

def read_video_metadata(video_path):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            f"Cannot open video: {video_path}"
        )

    fps = cap.get(
        cv2.CAP_PROP_FPS
    )

    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )

    cap.release()

    duration = (
        frame_count / fps
        if fps > 0
        else None
    )

    return {
        "path": str(video_path),
        "filename": video_path.name,
        "width": width,
        "height": height,
        "fps": fps,
        "frame_count": frame_count,
        "duration_sec": duration,
        "duration_min": (
            duration / 60
            if duration
            else None
        ),
    }


VIDEO_META = read_video_metadata(
    VIDEO_PATH
)


FRAME_WIDTH = VIDEO_META["width"]
FRAME_HEIGHT = VIDEO_META["height"]
FPS = VIDEO_META["fps"]
TOTAL_FRAMES = VIDEO_META["frame_count"]


print("=" * 70)
print("VIDEO")
print("=" * 70)

for key, value in VIDEO_META.items():
    print(f"{key:20s}: {value}")

VIDEO
path                : /kaggle/input/datasets/chrisbiran/traffic-tracker-videos/TOR3-PAGI.mp4
filename            : TOR3-PAGI.mp4
width               : 1280
height              : 720
fps                 : 29.81501660256117
frame_count         : 10282
duration_sec        : 344.85977777777777
duration_min        : 5.747662962962963


Define diagonal counting line

In [5]:
# ============================================================
# DIAGONAL COUNTING LINE
# ============================================================

COUNT_LINE = {
    "orientation": "diagonal",

    # Upper point
    "x1": int(FRAME_WIDTH * 0.95),
    "y1": int(FRAME_HEIGHT * 0.20),

    # Lower point
    "x2": int(FRAME_WIDTH * 0.05),
    "y2": int(FRAME_HEIGHT * 0.95),
}


print("=" * 70)
print("COUNTING LINE")
print("=" * 70)

print(
    f"Point 1: "
    f"({COUNT_LINE['x1']}, {COUNT_LINE['y1']})"
)

print(
    f"Point 2: "
    f"({COUNT_LINE['x2']}, {COUNT_LINE['y2']})"
)

COUNTING LINE
Point 1: (1216, 144)
Point 2: (64, 684)


Load YOLO26m

In [6]:
# ============================================================
# LOAD MODEL
# ============================================================

print("=" * 70)
print("LOADING MODEL")
print("=" * 70)

model = YOLO(
    MODEL_NAME
)

print(
    "Model loaded:",
    MODEL_NAME
)

print(
    "Classes:",
    len(model.names)
)

LOADING MODEL
Model loaded: yolo26m.pt
Classes: 80


Class mapping

In [7]:
# ============================================================
# CLASS MAPPING
# ============================================================

CLASS_NAMES = model.names

TARGET_CLASS_IDS = {
    class_id
    for class_id, class_name
    in CLASS_NAMES.items()
    if class_name in TARGET_CLASSES
}


print("=" * 70)
print("TARGET CLASSES")
print("=" * 70)

for class_id in sorted(
    TARGET_CLASS_IDS
):

    print(
        class_id,
        "->",
        CLASS_NAMES[class_id]
    )

TARGET CLASSES
0 -> person
2 -> car
3 -> motorcycle
5 -> bus
7 -> truck


PHASE 1: YOLO26m + BoT-SORT

In [8]:
print("=" * 70)
print("PHASE 1 — YOLO26m + BoT-SORT")
print("=" * 70)

start_time = time.time()

track_records = []

results_stream = model.track(
    source=str(VIDEO_PATH),

    tracker=TRACKER,

    imgsz=IMG_SIZE,

    conf=CONF_THRESHOLD,

    iou=IOU_THRESHOLD,

    device=DEVICE,

    stream=True,

    persist=True,

    verbose=False,
)


for frame_id, result in enumerate(
    results_stream,
    start=1
):

    boxes = result.boxes

    if (
        boxes is None
        or len(boxes) == 0
    ):
        continue


    # --------------------------------------------------------
    # Move tensors to CPU once
    # --------------------------------------------------------

    xyxy = (
        boxes.xyxy
        .cpu()
        .numpy()
    )

    conf = (
        boxes.conf
        .cpu()
        .numpy()
    )

    cls = (
        boxes.cls
        .cpu()
        .numpy()
        .astype(np.int16)
    )


    # --------------------------------------------------------
    # Track IDs
    # --------------------------------------------------------

    if boxes.id is None:
        continue

    track_ids = (
        boxes.id
        .cpu()
        .numpy()
        .astype(np.int32)
    )


    # --------------------------------------------------------
    # Filter target classes
    # --------------------------------------------------------

    mask = np.isin(
        cls,
        list(TARGET_CLASS_IDS)
    )

    if not np.any(mask):
        continue


    xyxy = xyxy[mask]
    conf = conf[mask]
    cls = cls[mask]
    track_ids = track_ids[mask]


    # --------------------------------------------------------
    # Vectorized bbox extraction
    # --------------------------------------------------------

    x1 = xyxy[:, 0]
    y1 = xyxy[:, 1]
    x2 = xyxy[:, 2]
    y2 = xyxy[:, 3]


    # --------------------------------------------------------
    # Bottom-center
    #
    # THIS is the trajectory point used by Phase 3.
    # --------------------------------------------------------

    bottom_center_x = (
        (x1 + x2) / 2.0
    )

    bottom_center_y = y2


    timestamp = (
        (frame_id - 1)
        / FPS
    )


    # --------------------------------------------------------
    # Build frame dataframe
    # --------------------------------------------------------

    frame_df = pd.DataFrame({

        "frame_id": frame_id,

        "timestamp_sec": timestamp,

        "track_id": track_ids,

        "class_id": cls,

        "class_name": [
            CLASS_NAMES[int(c)]
            for c in cls
        ],

        "confidence": conf,

        "x1": x1,

        "y1": y1,

        "x2": x2,

        "y2": y2,

        "bottom_center_x": bottom_center_x,

        "bottom_center_y": bottom_center_y,

    })


    track_records.append(
        frame_df
    )


# ------------------------------------------------------------
# Concatenate once
# ------------------------------------------------------------

tracks_raw = pd.concat(
    track_records,
    ignore_index=True
)


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

tracks_raw.sort_values(
    [
        "track_id",
        "frame_id"
    ],
    inplace=True
)


tracks_raw.reset_index(
    drop=True,
    inplace=True
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

tracks_raw.to_csv(
    PHASE1_TRACKS_PATH,
    index=False
)


elapsed = (
    time.time()
    -
    start_time
)


print("=" * 70)
print("PHASE 1 COMPLETE")
print("=" * 70)

print(
    f"Frames processed : {TOTAL_FRAMES:,}"
)

print(
    f"Tracking records : {len(tracks_raw):,}"
)

print(
    f"Unique tracks    : "
    f"{tracks_raw['track_id'].nunique():,}"
)

print(
    f"Elapsed          : "
    f"{elapsed:.2f} sec"
)

print(
    f"Processing FPS   : "
    f"{TOTAL_FRAMES / elapsed:.2f}"
)

print(
    "Output           :",
    PHASE1_TRACKS_PATH
)

PHASE 1 — YOLO26m + BoT-SORT
PHASE 1 COMPLETE
Frames processed : 10,282
Tracking records : 57,848
Unique tracks    : 815
Elapsed          : 542.98 sec
Processing FPS   : 18.94
Output           : /kaggle/working/traffic_counting/phase1/tracks_raw.csv


Reload Phase 1

In [9]:
# ============================================================
# LOAD PHASE 1 DATA
# ============================================================

tracks_raw = pd.read_csv(
    PHASE1_TRACKS_PATH
)

tracks_raw["track_id"] = (
    tracks_raw["track_id"]
    .astype(np.int32)
)

tracks_raw["frame_id"] = (
    tracks_raw["frame_id"]
    .astype(np.int32)
)

tracks_raw["class_name"] = (
    tracks_raw["class_name"]
    .astype("category")
)

print(
    tracks_raw.shape
)

display(
    tracks_raw.head()
)

(57848, 12)


,frame_id,timestamp_sec,track_id,class_id,class_name,confidence,x1,y1,x2,y2,bottom_center_x,bottom_center_y
0,1,0.000000,1,2,car,0.869134,177.05579,423.71393,216.99536,454.74396,197.02557,454.74396
1,2,0.033540,1,2,car,0.872478,177.63103,423.61182,217.95020,454.89438,197.79062,454.89438
2,3,0.067080,1,2,car,0.893524,178.32358,423.74670,219.28108,455.19150,198.80234,455.19150
3,4,0.100620,1,2,car,0.864520,178.90448,424.04730,220.36809,455.81708,199.63629,455.81708
4,5,0.134161,1,2,car,0.853626,179.52022,424.19507,221.41109,456.25223,200.46565,456.25223


# PHASE 2: Track-level class

1. Raw:
    a. class_name
    b. confidence
      ↓

2. Track-level:
    a. track_class
    b. track_class_ratio
    c. confidence_weighted_class
    d. class_ambiguous

In [10]:
# ============================================================
# MAJORITY CLASS COUNT
# ============================================================

class_counts = (
    tracks_raw
    .groupby(
        [
            "track_id",
            "class_name"
        ],
        observed=True
    )
    .size()
    .rename("class_observations")
    .reset_index()
)


# ============================================================
# TOTAL OBSERVATIONS PER TRACK
# ============================================================

track_totals = (
    class_counts
    .groupby("track_id")[
        "class_observations"
    ]
    .sum()
    .rename("total_observations")
)


class_counts = (
    class_counts
    .join(
        track_totals,
        on="track_id"
    )
)


class_counts["class_ratio"] = (
    class_counts["class_observations"]
    /
    class_counts["total_observations"]
)


# ============================================================
# MAJORITY CLASS
# ============================================================

majority_class = (
    class_counts
    .sort_values(
        [
            "track_id",
            "class_observations",
            "class_ratio"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .drop_duplicates(
        "track_id"
    )
    [
        [
            "track_id",
            "class_name",
            "class_ratio"
        ]
    ]
    .rename(
        columns={
            "class_name":
                "track_class",

            "class_ratio":
                "track_class_ratio"
        }
    )
)


# ============================================================
# CONFIDENCE-WEIGHTED CLASS
# ============================================================

weighted_votes = (
    tracks_raw
    .groupby(
        [
            "track_id",
            "class_name"
        ],
        observed=True
    )["confidence"]
    .sum()
    .rename(
        "weighted_confidence"
    )
    .reset_index()
)


weighted_class = (
    weighted_votes
    .sort_values(
        [
            "track_id",
            "weighted_confidence"
        ],
        ascending=[
            True,
            False
        ]
    )
    .drop_duplicates(
        "track_id"
    )
    [
        [
            "track_id",
            "class_name"
        ]
    ]
    .rename(
        columns={
            "class_name":
                "confidence_weighted_class"
        }
    )
)


# ============================================================
# CLASS AMBIGUITY
#
# Conservative:
# If majority class ratio < 70%,
# mark ambiguous.
# ============================================================

majority_class["class_ambiguous"] = (
    majority_class["track_class_ratio"]
    < 0.70
)


# ============================================================
# MERGE
# ============================================================

track_level = (
    majority_class
    .merge(
        weighted_class,
        on="track_id",
        how="left"
    )
)


print("=" * 70)
print("TRACK-LEVEL CLASS")
print("=" * 70)

display(
    track_level.head(20)
)

TRACK-LEVEL CLASS


,track_id,track_class,track_class_ratio,class_ambiguous,confidence_weighted_class
0,1,car,1.000000,False,car
1,2,car,1.000000,False,car
2,4,motorcycle,1.000000,False,motorcycle
3,18,truck,1.000000,False,truck
4,22,motorcycle,0.807229,False,motorcycle
5,33,motorcycle,0.600000,True,motorcycle
6,41,person,1.000000,False,person
7,47,motorcycle,1.000000,False,motorcycle
8,51,person,1.000000,False,person
9,67,person,1.000000,False,person


Track observation coverage

In [11]:
# ============================================================
# TRACK COVERAGE
# ============================================================

track_frame_stats = (
    tracks_raw
    .groupby("track_id")
    .agg(
        first_frame=(
            "frame_id",
            "min"
        ),

        last_frame=(
            "frame_id",
            "max"
        ),

        observed_frames=(
            "frame_id",
            "nunique"
        ),

        mean_confidence=(
            "confidence",
            "mean"
        ),

        median_confidence=(
            "confidence",
            "median"
        ),
    )
)


track_frame_stats["expected_frames"] = (
    track_frame_stats["last_frame"]
    -
    track_frame_stats["first_frame"]
    +
    1
)


track_frame_stats["observation_ratio"] = (
    track_frame_stats["observed_frames"]
    /
    track_frame_stats["expected_frames"]
)


track_frame_stats["duration_sec"] = (
    track_frame_stats["expected_frames"]
    /
    FPS
)


track_level = (
    track_level
    .merge(
        track_frame_stats,
        on="track_id",
        how="left"
    )
)


display(
    track_level.head()
)

,track_id,track_class,track_class_ratio,class_ambiguous,confidence_weighted_class,first_frame,last_frame,observed_frames,mean_confidence,median_confidence,expected_frames,observation_ratio,duration_sec
0,1,car,1.000000,False,car,1,91,91,0.881578,0.901310,91,1.000000,3.052153
1,2,car,1.000000,False,car,1,319,231,0.365587,0.361963,319,0.724138,10.699306
2,4,motorcycle,1.000000,False,motorcycle,11,4424,4091,0.570487,0.576808,4414,0.926824,148.046203
3,18,truck,1.000000,False,truck,50,58,4,0.289544,0.275458,9,0.444444,0.301861
4,22,motorcycle,0.807229,False,motorcycle,62,151,83,0.531693,0.435839,90,0.922222,3.018613


Gap analysis

In [12]:
# ============================================================
# GAP ANALYSIS
# ============================================================

tracks_raw["frame_gap"] = (
    tracks_raw
    .groupby("track_id")["frame_id"]
    .diff()
)


tracks_raw["gap_sec"] = (
    tracks_raw["frame_gap"]
    /
    FPS
)


gap_mask = (
    tracks_raw["gap_sec"]
    > 1
)


gap_summary = (
    tracks_raw.loc[gap_mask]
    .groupby("track_id")
    .agg(
        gap_events=(
            "gap_sec",
            "size"
        ),

        max_gap_sec=(
            "gap_sec",
            "max"
        ),

        total_gap_sec=(
            "gap_sec",
            "sum"
        ),
    )
)


track_level = (
    track_level
    .merge(
        gap_summary,
        on="track_id",
        how="left"
    )
)


track_level[
    [
        "gap_events",
        "max_gap_sec",
        "total_gap_sec"
    ]
] = (
    track_level[
        [
            "gap_events",
            "max_gap_sec",
            "total_gap_sec"
        ]
    ]
    .fillna(0)
)

Track quality score

In [13]:
# ============================================================
# TRACK QUALITY
# ============================================================

track_level["quality_score"] = (
    0.35
    * track_level["track_class_ratio"]
    +
    0.30
    * track_level["observation_ratio"].clip(0, 1)
    +
    0.35
    * track_level["mean_confidence"].clip(0, 1)
)


# ============================================================
# QUALITY LEVEL
# ============================================================

track_level["quality_level"] = pd.cut(
    track_level["quality_score"],
    bins=[
        -np.inf,
        0.60,
        0.75,
        0.90,
        np.inf
    ],
    labels=[
        "poor",
        "moderate",
        "good",
        "excellent"
    ],
    right=False
)


# ============================================================
# SAVE
# ============================================================

track_level.to_csv(
    PHASE2_SUMMARY_PATH,
    index=False
)


print("=" * 70)
print("PHASE 2 — TRACK QUALITY")
print("=" * 70)

display(
    track_level[
        [
            "track_id",
            "track_class",
            "track_class_ratio",
            "confidence_weighted_class",
            "class_ambiguous",
            "observation_ratio",
            "mean_confidence",
            "quality_score",
            "quality_level"
        ]
    ].head(20)
)

PHASE 2 — TRACK QUALITY


,track_id,track_class,track_class_ratio,confidence_weighted_class,class_ambiguous,observation_ratio,mean_confidence,quality_score,quality_level
0,1,car,1.000000,car,False,1.000000,0.881578,0.958552,excellent
1,2,car,1.000000,car,False,0.724138,0.365587,0.695197,moderate
2,4,motorcycle,1.000000,motorcycle,False,0.926824,0.570487,0.827718,good
3,18,truck,1.000000,truck,False,0.444444,0.289544,0.584674,poor
4,22,motorcycle,0.807229,motorcycle,False,0.922222,0.531693,0.745289,moderate
5,33,motorcycle,0.600000,motorcycle,True,0.833333,0.414032,0.604911,moderate
6,41,person,1.000000,person,False,1.000000,0.480055,0.818019,good
7,47,motorcycle,1.000000,motorcycle,False,1.000000,0.470788,0.814776,good
8,51,person,1.000000,person,False,1.000000,0.352660,0.773431,good
9,67,person,1.000000,person,False,1.000000,0.685627,0.889970,good


Merge track-level class back to every observation

In [14]:
# ============================================================
# ATTACH TRACK-LEVEL CLASS
#
# IMPORTANT:
#
# Phase 3 will use ONLY:
#
#   track_class
#   track_class_ratio
#   class_ambiguous
#
# It will NOT use raw class_name.
# ============================================================

tracks_phase2 = (
    tracks_raw
    .merge(
        track_level[
            [
                "track_id",
                "track_class",
                "track_class_ratio",
                "class_ambiguous",
                "quality_score",
                "quality_level"
            ]
        ],
        on="track_id",
        how="left",
        validate="many_to_one"
    )
)


tracks_phase2.to_csv(
    PHASE2_TRACKS_PATH,
    index=False
)


print(
    "Output:",
    PHASE2_TRACKS_PATH
)

display(
    tracks_phase2.head()
)

Output: /kaggle/working/traffic_counting/phase2/tracks_with_track_class.csv


,frame_id,timestamp_sec,track_id,class_id,class_name,confidence,x1,y1,x2,y2,bottom_center_x,bottom_center_y,frame_gap,gap_sec,track_class,track_class_ratio,class_ambiguous,quality_score,quality_level
0,1,0.000000,1,2,car,0.869134,177.05579,423.71393,216.99536,454.74396,197.02557,454.74396,NaN,NaN,car,1.0,False,0.958552,excellent
1,2,0.033540,1,2,car,0.872478,177.63103,423.61182,217.95020,454.89438,197.79062,454.89438,1.0,0.03354,car,1.0,False,0.958552,excellent
2,3,0.067080,1,2,car,0.893524,178.32358,423.74670,219.28108,455.19150,198.80234,455.19150,1.0,0.03354,car,1.0,False,0.958552,excellent
3,4,0.100620,1,2,car,0.864520,178.90448,424.04730,220.36809,455.81708,199.63629,455.81708,1.0,0.03354,car,1.0,False,0.958552,excellent
4,5,0.134161,1,2,car,0.853626,179.52022,424.19507,221.41109,456.25223,200.46565,456.25223,1.0,0.03354,car,1.0,False,0.958552,excellent


Verify Phase 2

In [15]:
# ============================================================
# PHASE 2 VALIDATION
# ============================================================

print("=" * 70)
print("PHASE 2 VALIDATION")
print("=" * 70)


print(
    "Unique tracks:",
    track_level["track_id"].nunique()
)


print(
    "Track-level classes:"
)

display(
    track_level[
        "track_class"
    ]
    .value_counts()
)


print(
    "\nAmbiguous tracks:"
)

print(
    track_level[
        "class_ambiguous"
    ]
    .value_counts()
)


print(
    "\nQuality:"
)

display(
    track_level[
        "quality_level"
    ]
    .value_counts()
    .rename("track_count")
    .to_frame()
)

PHASE 2 VALIDATION
Unique tracks: 815
Track-level classes:


track_class
person        393
motorcycle    238
car           142
truck          41
bus             1
Name: count, dtype: int64


Ambiguous tracks:
class_ambiguous
False    769
True      46
Name: count, dtype: int64

Quality:


,track_count
quality_level,
good,473
moderate,240
poor,61
excellent,41


# PHASE 3: Bottom-center trajectory

In [16]:
# ============================================================
# BOTTOM-CENTER TRAJECTORY
# ============================================================

trajectory = (
    tracks_phase2[
        [
            "track_id",
            "frame_id",
            "timestamp_sec",
            "bottom_center_x",
            "bottom_center_y",
            "track_class",
            "track_class_ratio",
            "class_ambiguous"
        ]
    ]
    .sort_values(
        [
            "track_id",
            "frame_id"
        ]
    )
    .copy()
)


trajectory["dx"] = (
    trajectory
    .groupby("track_id")[
        "bottom_center_x"
    ]
    .diff()
)


trajectory["dy"] = (
    trajectory
    .groupby("track_id")[
        "bottom_center_y"
    ]
    .diff()
)


trajectory["frame_delta"] = (
    trajectory
    .groupby("track_id")[
        "frame_id"
    ]
    .diff()
)


trajectory["time_delta_sec"] = (
    trajectory["frame_delta"]
    /
    FPS
)


print(
    trajectory.shape
)

display(
    trajectory.head(20)
)

(57848, 12)


,track_id,frame_id,timestamp_sec,bottom_center_x,bottom_center_y,track_class,track_class_ratio,class_ambiguous,dx,dy,frame_delta,time_delta_sec
0,1,1,0.000000,197.02557,454.74396,car,1.0,False,NaN,NaN,NaN,NaN
1,1,2,0.033540,197.79062,454.89438,car,1.0,False,0.76505,0.15042,1.0,0.03354
2,1,3,0.067080,198.80234,455.19150,car,1.0,False,1.01172,0.29712,1.0,0.03354
3,1,4,0.100620,199.63629,455.81708,car,1.0,False,0.83395,0.62558,1.0,0.03354
4,1,5,0.134161,200.46565,456.25223,car,1.0,False,0.82936,0.43515,1.0,0.03354
5,1,6,0.167701,201.64090,456.52036,car,1.0,False,1.17525,0.26813,1.0,0.03354
6,1,7,0.201241,202.63475,456.79675,car,1.0,False,0.99385,0.27639,1.0,0.03354
7,1,8,0.234781,203.44376,457.38144,car,1.0,False,0.80901,0.58469,1.0,0.03354
8,1,9,0.268321,203.39865,458.02770,car,1.0,False,-0.04511,0.64626,1.0,0.03354
9,1,10,0.301861,204.65872,458.43167,car,1.0,False,1.26007,0.40397,1.0,0.03354


Geometry of diagonal line

In [17]:
# ============================================================
# LINE GEOMETRY
# ============================================================

X1 = COUNT_LINE["x1"]
Y1 = COUNT_LINE["y1"]

X2 = COUNT_LINE["x2"]
Y2 = COUNT_LINE["y2"]


LINE_DX = X2 - X1
LINE_DY = Y2 - Y1


LINE_LENGTH = math.hypot(
    LINE_DX,
    LINE_DY
)


def signed_line_value(
    x,
    y
):
    """
    Signed cross-product value.

    > 0 = one side
    < 0 = opposite side
    = 0 = exactly on line
    """

    return (
        LINE_DX * (y - Y1)
        -
        LINE_DY * (x - X1)
    )


trajectory["line_value"] = (
    LINE_DX
    * (
        trajectory["bottom_center_y"]
        -
        Y1
    )
    -
    LINE_DY
    * (
        trajectory["bottom_center_x"]
        -
        X1
    )
)


# ------------------------------------------------------------
# Normalize to approximate pixel distance from line
# ------------------------------------------------------------

trajectory["line_distance_px"] = (
    np.abs(
        trajectory["line_value"]
    )
    /
    LINE_LENGTH
)


# ------------------------------------------------------------
# Deadband
# ------------------------------------------------------------

trajectory["side"] = np.select(
    [
        trajectory["line_distance_px"]
        <= LINE_DEADBAND_PX,

        trajectory["line_value"] > 0
    ],
    [
        0,
        1
    ],
    default=-1
)


display(
    trajectory[
        [
            "track_id",
            "frame_id",
            "bottom_center_x",
            "bottom_center_y",
            "line_distance_px",
            "side"
        ]
    ].head(20)
)

,track_id,frame_id,bottom_center_x,bottom_center_y,line_distance_px,side
0,1,1,197.02557,454.74396,151.121372,1
1,1,2,197.79062,454.89438,150.660459,1
2,1,3,198.80234,455.19150,149.962021,1
3,1,4,199.63629,455.81708,149.041628,1
4,1,5,200.46565,456.25223,148.295609,1
5,1,6,201.64090,456.52036,147.554012,1
6,1,7,202.63475,456.79675,146.881929,1
7,1,8,203.44376,457.38144,146.009145,1
8,1,9,203.39865,458.02770,145.443129,1
9,1,10,204.65872,458.43167,144.542534,1


Remove observations too close to line

In [18]:
# ============================================================
# STABLE SIDE OBSERVATIONS
# ============================================================

trajectory_stable = trajectory[
    trajectory["side"] != 0
].copy()


trajectory_stable["previous_side"] = (
    trajectory_stable
    .groupby("track_id")["side"]
    .shift(1)
)


trajectory_stable["previous_frame"] = (
    trajectory_stable
    .groupby("track_id")["frame_id"]
    .shift(1)
)


trajectory_stable["frame_gap"] = (
    trajectory_stable["frame_id"]
    -
    trajectory_stable["previous_frame"]
)


# ------------------------------------------------------------
# Valid consecutive observations
# ------------------------------------------------------------

trajectory_stable["valid_temporal_transition"] = (
    trajectory_stable["frame_gap"]
    <= (
        MAX_TRAJECTORY_GAP_SEC
        * FPS
    )
)


trajectory_stable["crossed"] = (
    trajectory_stable["valid_temporal_transition"]
    &
    trajectory_stable["previous_side"].notna()
    &
    (
        trajectory_stable["side"]
        !=
        trajectory_stable["previous_side"]
    )
)


crossing_candidates = (
    trajectory_stable[
        trajectory_stable["crossed"]
    ]
    .copy()
)


print(
    "Crossing candidates:",
    len(crossing_candidates)
)

Crossing candidates: 157


Convert crossing candidates into events

In [19]:
# ============================================================
# CROSSING EVENTS
# ============================================================

crossing_candidates["direction"] = np.where(
    crossing_candidates["previous_side"] < 0,
    "side_-1_to_+1",
    "side_+1_to_-1"
)


# ============================================================
# ONE CROSSING / TRACK
#
# If a track crosses multiple times because of jitter,
# keep the FIRST valid crossing only.
# ============================================================

crossing_events = (
    crossing_candidates
    .sort_values(
        [
            "track_id",
            "frame_id"
        ]
    )
    .drop_duplicates(
        subset=["track_id"],
        keep="first"
    )
    [
        [
            "track_id",
            "frame_id",
            "timestamp_sec",
            "bottom_center_x",
            "bottom_center_y",
            "direction",
            "track_class",
            "track_class_ratio",
            "class_ambiguous"
        ]
    ]
    .rename(
        columns={
            "frame_id":
                "crossing_frame",

            "timestamp_sec":
                "crossing_time_sec",

            "bottom_center_x":
                "crossing_x",

            "bottom_center_y":
                "crossing_y"
        }
    )
    .reset_index(drop=True)
)


print("=" * 70)
print("CROSSING EVENTS")
print("=" * 70)

print(
    "Unique crossing tracks:",
    len(crossing_events)
)

display(
    crossing_events.head(20)
)

CROSSING EVENTS
Unique crossing tracks: 155


,track_id,crossing_frame,crossing_time_sec,crossing_x,crossing_y,direction,track_class,track_class_ratio,class_ambiguous
0,1,70,2.314270,434.04553,521.81170,side_+1_to_-1,car,1.000000,False
1,22,134,4.460839,443.27124,515.99536,side_+1_to_-1,motorcycle,0.807229,False
2,67,137,4.561460,481.67487,499.22678,side_+1_to_-1,person,1.000000,False
3,185,497,16.635912,312.97046,548.70044,side_-1_to_+1,motorcycle,1.000000,False
4,192,495,16.568832,340.83755,531.81830,side_-1_to_+1,person,0.921875,False
5,220,700,23.444562,438.70264,519.48130,side_+1_to_-1,motorcycle,1.000000,False
6,275,706,23.645803,471.77246,508.91483,side_+1_to_-1,person,1.000000,False
7,321,771,25.825912,318.93054,553.99190,side_-1_to_+1,truck,0.996241,False
8,398,879,29.448248,437.52527,523.81690,side_+1_to_-1,motorcycle,1.000000,False
9,403,882,29.548868,472.82916,510.88724,side_+1_to_-1,person,1.000000,False


Remove person

In [20]:
# ============================================================
# PERSON EXCLUSION
#
# PERSON NEVER ENTERS VEHICLE COUNT.
#
# Motorcycle + rider:
#     motorcycle = 1 vehicle
#
# Person:
#     pedestrian only
# ============================================================

crossing_vehicle = crossing_events[
    crossing_events["track_class"].isin(
        VEHICLE_CLASSES
    )
].copy()


crossing_person = crossing_events[
    crossing_events["track_class"]
    ==
    "person"
].copy()


print("=" * 70)
print("PERSON EXCLUSION")
print("=" * 70)

print(
    "Total crossing events:",
    len(crossing_events)
)

print(
    "Vehicle crossings:",
    len(crossing_vehicle)
)

print(
    "Person crossings:",
    len(crossing_person)
)

PERSON EXCLUSION
Total crossing events: 155
Vehicle crossings: 86
Person crossings: 69


Conservative motorcycle fragmentation suppression

In [21]:
# ============================================================
# MOTORCYCLE FRAGMENTATION DEDUP
#
# CONSERVATIVE
# ============================================================

vehicle_events = (
    crossing_vehicle
    .sort_values(
        "crossing_time_sec"
    )
    .reset_index(
        drop=True
    )
)


vehicle_events["duplicate_of_track_id"] = pd.NA

vehicle_events["dedup_reason"] = ""


# ------------------------------------------------------------
# Only process motorcycles
# ------------------------------------------------------------

motorcycle_idx = np.flatnonzero(
    (
        vehicle_events["track_class"]
        ==
        "motorcycle"
    ).to_numpy()
)


accepted_motorcycles = []


for idx in motorcycle_idx:

    current = vehicle_events.iloc[idx]


    # --------------------------------------------------------
    # Compare only against previously accepted motorcycles
    # --------------------------------------------------------

    duplicate_found = False

    duplicate_track_id = None


    for accepted_idx in accepted_motorcycles:

        previous = (
            vehicle_events
            .iloc[accepted_idx]
        )


        # ----------------------------------------------------
        # Same direction required
        # ----------------------------------------------------

        if (
            current["direction"]
            !=
            previous["direction"]
        ):
            continue


        # ----------------------------------------------------
        # Temporal proximity
        # ----------------------------------------------------

        dt = abs(
            current["crossing_time_sec"]
            -
            previous["crossing_time_sec"]
        )


        if dt > MOTO_DEDUP_TIME_SEC:
            continue


        # ----------------------------------------------------
        # Spatial proximity
        # ----------------------------------------------------

        distance = math.hypot(
            current["crossing_x"]
            -
            previous["crossing_x"],

            current["crossing_y"]
            -
            previous["crossing_y"]
        )


        if distance > MOTO_DEDUP_DISTANCE_PX:
            continue


        # ----------------------------------------------------
        # Candidate duplicate
        # ----------------------------------------------------

        duplicate_found = True

        duplicate_track_id = (
            previous["track_id"]
        )

        break


    if duplicate_found:

        vehicle_events.at[
            idx,
            "duplicate_of_track_id"
        ] = duplicate_track_id

        vehicle_events.at[
            idx,
            "dedup_reason"
        ] = (
            "motorcycle_fragmentation"
        )

    else:

        accepted_motorcycles.append(
            idx
        )


# ------------------------------------------------------------
# Final accepted events
# ------------------------------------------------------------

vehicle_events["is_duplicate"] = (
    vehicle_events[
        "duplicate_of_track_id"
    ]
    .notna()
)


final_crossings = vehicle_events[
    ~vehicle_events["is_duplicate"]
].copy()


print("=" * 70)
print("MOTORCYCLE DEDUP")
print("=" * 70)

print(
    "Before dedup:",
    len(vehicle_events)
)

print(
    "Duplicates:",
    vehicle_events[
        "is_duplicate"
    ].sum()
)

print(
    "After dedup:",
    len(final_crossings)
)

MOTORCYCLE DEDUP
Before dedup: 86
Duplicates: 11
After dedup: 75


Final vehicle count

In [22]:
# ============================================================
# FINAL VEHICLE COUNT
# ============================================================

final_counts = (
    final_crossings
    .groupby(
        "track_class"
    )
    .size()
    .reindex(
        [
            "motorcycle",
            "car",
            "truck",
            "bus"
        ],
        fill_value=0
    )
    .rename(
        "vehicle_count"
    )
    .reset_index()
)


final_counts["vehicle_count"] = (
    final_counts["vehicle_count"]
    .astype(int)
)


total_vehicle_count = int(
    final_counts[
        "vehicle_count"
    ].sum()
)


print("=" * 70)
print("FINAL VEHICLE COUNT")
print("=" * 70)

display(
    final_counts
)


print(
    "TOTAL VEHICLES:",
    total_vehicle_count
)

FINAL VEHICLE COUNT


,track_class,vehicle_count
0,motorcycle,59
1,car,7
2,truck,9
3,bus,0


TOTAL VEHICLES: 75


Save crossing events

In [23]:
# ============================================================
# SAVE CROSSING EVENTS
# ============================================================

crossing_events.to_csv(
    PHASE3_CROSSINGS_PATH,
    index=False
)


final_crossings.to_csv(
    PHASE3_DIR /
    "final_vehicle_crossings.csv",
    index=False
)


final_counts.to_csv(
    PHASE3_FINAL_COUNTS_PATH,
    index=False
)


print(
    "Saved:"
)

print(
    PHASE3_CROSSINGS_PATH
)

print(
    PHASE3_FINAL_COUNTS_PATH
)

Saved:
/kaggle/working/traffic_counting/phase3/crossing_events.csv
/kaggle/working/traffic_counting/phase3/final_vehicle_counts.csv


Phase 3 audit

In [24]:
# ============================================================
# PHASE 3 AUDIT
# ============================================================

audit = {
    "all_crossing_events":
        len(crossing_events),

    "person_crossings_excluded":
        len(crossing_person),

    "vehicle_crossings_before_dedup":
        len(vehicle_events),

    "motorcycle_duplicates_removed":
        int(
            vehicle_events[
                "is_duplicate"
            ].sum()
        ),

    "final_vehicle_crossings":
        len(final_crossings),

    "final_vehicle_count":
        total_vehicle_count,
}


print("=" * 70)
print("PHASE 3 AUDIT")
print("=" * 70)

for key, value in audit.items():

    print(
        f"{key:35s}: {value:,}"
    )


with open(
    PHASE3_DIR /
    "phase3_audit.json",
    "w"
) as f:

    json.dump(
        audit,
        f,
        indent=4
    )

PHASE 3 AUDIT
all_crossing_events                : 155
person_crossings_excluded          : 69
vehicle_crossings_before_dedup     : 86
motorcycle_duplicates_removed      : 11
final_vehicle_crossings            : 75
final_vehicle_count                : 75


Count by direction

In [25]:
# ============================================================
# COUNT BY DIRECTION
# ============================================================

direction_counts = (
    final_crossings
    .groupby(
        [
            "track_class",
            "direction"
        ]
    )
    .size()
    .rename(
        "count"
    )
    .reset_index()
)


display(
    direction_counts
)

,track_class,direction,count
0,bus,side_+1_to_-1,0
1,bus,side_-1_to_+1,0
2,car,side_+1_to_-1,3
3,car,side_-1_to_+1,4
4,motorcycle,side_+1_to_-1,26
5,motorcycle,side_-1_to_+1,33
6,person,side_+1_to_-1,0
7,person,side_-1_to_+1,0
8,truck,side_+1_to_-1,3
9,truck,side_-1_to_+1,6


Count by track quality

In [26]:
# ============================================================
# COUNT BY TRACK QUALITY
# ============================================================

crossing_quality = (
    final_crossings
    .merge(
        track_level[
            [
                "track_id",
                "quality_score",
                "quality_level"
            ]
        ],
        on="track_id",
        how="left",
        validate="one_to_one"
    )
)


quality_count = (
    crossing_quality
    .groupby(
        [
            "track_class",
            "quality_level"
        ],
        observed=True
    )
    .size()
    .rename(
        "count"
    )
    .reset_index()
)


display(
    quality_count
)

,track_class,quality_level,count
0,car,poor,1
1,car,moderate,2
2,car,good,1
3,car,excellent,3
4,motorcycle,moderate,4
5,motorcycle,good,51
6,motorcycle,excellent,4
7,truck,moderate,2
8,truck,good,7


Prepare frame lookup

In [27]:
# ============================================================
# VIDEO RENDER PREPARATION
# ============================================================

# Only observations needed for visualization
render_df = tracks_phase2[
    [
        "frame_id",
        "track_id",
        "x1",
        "y1",
        "x2",
        "y2",
        "track_class",
        "track_class_ratio",
        "class_ambiguous"
    ]
].copy()


render_df["frame_id"] = (
    render_df["frame_id"]
    .astype(np.int32)
)


# ------------------------------------------------------------
# Group by frame once
# ------------------------------------------------------------

frame_groups = {
    frame_id: group
    for frame_id, group
    in render_df.groupby(
        "frame_id",
        sort=False
    )
}


# ------------------------------------------------------------
# Crossing events by frame
# ------------------------------------------------------------

crossing_frame_groups = {
    frame_id: group
    for frame_id, group
    in final_crossings.groupby(
        "crossing_frame",
        sort=False
    )
}


# ------------------------------------------------------------
# Cumulative count lookup
#
# IMPORTANT:
# This uses INTEGER frame keys.
#
# No tuple-key bug.
# ------------------------------------------------------------

crossing_frame_counts = (
    final_crossings[
        "crossing_frame"
    ]
    .value_counts()
    .sort_index()
)


cumulative_count_by_frame = (
    crossing_frame_counts
    .reindex(
        range(
            1,
            TOTAL_FRAMES + 1
        ),
        fill_value=0
    )
    .cumsum()
    .astype(int)
)


print(
    "Frame groups:",
    len(frame_groups)
)

print(
    "Crossing frames:",
    len(crossing_frame_groups)
)

print(
    "Final count:",
    cumulative_count_by_frame.iloc[-1]
)

Frame groups: 10282
Crossing frames: 74
Final count: 75


FFmpeg availability

In [28]:
# ============================================================
# FFMPEG CHECK
# ============================================================

ffmpeg_path = (
    subprocess
    .check_output(
        [
            "which",
            "ffmpeg"
        ],
        text=True
    )
    .strip()
)


print(
    "FFmpeg:",
    ffmpeg_path
)


version = subprocess.check_output(
    [
        "ffmpeg",
        "-version"
    ],
    text=True
)


print(
    version.splitlines()[0]
)

FFmpeg: /usr/bin/ffmpeg
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


In [29]:
# ============================================================
# VIDEO RENDER HELPERS
# ============================================================

def draw_counting_line(
    frame
):

    cv2.line(
        frame,

        (
            COUNT_LINE["x1"],
            COUNT_LINE["y1"]
        ),

        (
            COUNT_LINE["x2"],
            COUNT_LINE["y2"]
        ),

        (0, 255, 255),

        4,

        cv2.LINE_AA
    )


    # Endpoints

    cv2.circle(
        frame,

        (
            COUNT_LINE["x1"],
            COUNT_LINE["y1"]
        ),

        7,

        (255, 0, 0),

        -1
    )


    cv2.circle(
        frame,

        (
            COUNT_LINE["x2"],
            COUNT_LINE["y2"]
        ),

        7,

        (255, 0, 0),

        -1
    )


    cv2.putText(
        frame,

        "COUNTING LINE",

        (
            min(
                COUNT_LINE["x1"] + 10,
                FRAME_WIDTH - 200
            ),

            max(
                COUNT_LINE["y1"] - 12,
                25
            )
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.65,

        (0, 255, 255),

        2,

        cv2.LINE_AA
    )


    return frame


def draw_track(
    frame,
    row
):

    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    track_id = int(
        row["track_id"]
    )

    # ========================================================
    # IMPORTANT:
    # USE ONLY TRACK-LEVEL CLASS
    # ========================================================

    track_class = str(
        row["track_class"]
    )

    ratio = float(
        row["track_class_ratio"]
    )

    ambiguous = bool(
        row["class_ambiguous"]
    )


    label = (
        f"ID {track_id} | "
        f"{track_class.upper()} | "
        f"{ratio:.0%}"
    )


    if ambiguous:

        label += " | AMBIG"


    # --------------------------------------------------------
    # Bounding box
    # --------------------------------------------------------

    cv2.rectangle(
        frame,

        (x1, y1),

        (x2, y2),

        (0, 255, 0),

        2
    )


    # --------------------------------------------------------
    # Label
    # --------------------------------------------------------

    (
        text_w,
        text_h
    ), baseline = cv2.getTextSize(
        label,
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        1
    )


    text_x = x1

    text_y = max(
        text_h + baseline + 2,
        y1 - 5
    )


    cv2.rectangle(
        frame,

        (
            text_x,
            text_y
            -
            text_h
            -
            baseline
        ),

        (
            text_x
            +
            text_w
            +
            4,

            text_y
            +
            2
        ),

        (0, 0, 0),

        -1
    )


    cv2.putText(
        frame,

        label,

        (
            text_x + 2,
            text_y
        ),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.45,

        (255, 255, 255),

        1,

        cv2.LINE_AA
    )


    return frame

In [31]:
# ============================================================
# PHASE 3 — PREPARE CUMULATIVE COUNT ARRAY
#
# PURPOSE
# ------------------------------------------------------------
# Convert cumulative_count_by_frame into a plain NumPy array.
#
# WHY?
# ------------------------------------------------------------
# Pandas Series uses LABEL-based indexing with [].
# Rendering requires POSITION-based indexing.
#
# Therefore:
#
# Pandas Series
#      ↓
# reset_index(drop=True)
#      ↓
# NumPy array
#      ↓
# cumulative_count_array[frame_id - 1]
#
# This makes rendering independent from Pandas index.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 70)
print("PREPARING CUMULATIVE COUNT ARRAY")
print("=" * 70)


# ============================================================
# INSPECT ORIGINAL OBJECT
# ============================================================

print(
    "Original type :",
    type(cumulative_count_by_frame)
)


if isinstance(
    cumulative_count_by_frame,
    pd.Series
):

    print(
        "Original index type :",
        type(
            cumulative_count_by_frame.index
        )
    )

    print(
        "Original length     :",
        len(
            cumulative_count_by_frame
        )
    )

    print(
        "Original first index:",
        cumulative_count_by_frame.index[:5].tolist()
    )


# ============================================================
# CONVERT TO NUMPY
# ============================================================

if isinstance(
    cumulative_count_by_frame,
    pd.Series
):

    cumulative_count_array = (
        cumulative_count_by_frame
        .reset_index(drop=True)
        .to_numpy(
            dtype=np.int64
        )
    )

else:

    cumulative_count_array = np.asarray(
        cumulative_count_by_frame,
        dtype=np.int64
    )


# ============================================================
# VALIDATION
# ============================================================

print(
    "\nConverted type:",
    type(cumulative_count_array)
)

print(
    "Length:",
    len(cumulative_count_array)
)


# ============================================================
# VIDEO FRAME VALIDATION
# ============================================================

if "TOTAL_FRAMES" not in globals():

    cap_check = cv2.VideoCapture(
        str(VIDEO_PATH)
    )

    if not cap_check.isOpened():

        raise RuntimeError(
            f"Cannot open video: {VIDEO_PATH}"
        )

    TOTAL_FRAMES = int(
        cap_check.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    cap_check.release()


print(
    "Video frames:",
    TOTAL_FRAMES
)


# ============================================================
# IMPORTANT
# ============================================================
#
# We allow either:
#
# len == TOTAL_FRAMES
#
# OR
#
# len == TOTAL_FRAMES + 1
#
# depending on how the cumulative series was constructed.
#
# ============================================================

if len(cumulative_count_array) == TOTAL_FRAMES:

    pass


elif len(cumulative_count_array) == TOTAL_FRAMES + 1:

    print(
        "\nDetected TOTAL_FRAMES + 1 structure."
    )

    # Remove initial frame-0 state.
    cumulative_count_array = (
        cumulative_count_array[1:]
    )


else:

    raise ValueError(
        "\nCumulative count length does not "
        "match video frame count.\n\n"
        f"Video frames       : {TOTAL_FRAMES:,}\n"
        f"Cumulative length  : "
        f"{len(cumulative_count_array):,}\n\n"
        "Rebuild cumulative_count_by_frame "
        "before rendering."
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

assert len(
    cumulative_count_array
) == TOTAL_FRAMES


# ============================================================
# SANITY CHECK
# ============================================================

if np.any(
    cumulative_count_array < 0
):

    raise ValueError(
        "Cumulative vehicle count contains "
        "negative values."
    )


if len(
    cumulative_count_array
) > 1:

    if np.any(
        np.diff(
            cumulative_count_array
        ) < 0
    ):

        raise ValueError(
            "Cumulative count is decreasing. "
            "This should never happen."
        )


print("\n" + "=" * 70)
print("CUMULATIVE COUNT ARRAY READY")
print("=" * 70)

print(
    f"Length        : "
    f"{len(cumulative_count_array):,}"
)

print(
    f"First values  : "
    f"{cumulative_count_array[:10].tolist()}"
)

print(
    f"Last values   : "
    f"{cumulative_count_array[-10:].tolist()}"
)

print(
    f"Final count   : "
    f"{int(cumulative_count_array[-1])}"
)

print(
    "Validation    : PASS"
)

PREPARING CUMULATIVE COUNT ARRAY
Original type : <class 'pandas.core.series.Series'>
Original index type : <class 'pandas.core.indexes.range.RangeIndex'>
Original length     : 10282
Original first index: [1, 2, 3, 4, 5]

Converted type: <class 'numpy.ndarray'>
Length: 10282
Video frames: 10282

CUMULATIVE COUNT ARRAY READY
Length        : 10,282
First values  : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Last values   : [75, 75, 75, 75, 75, 75, 75, 75, 75, 75]
Final count   : 75
Validation    : PASS


In [35]:
# ============================================================
# PHASE 3 — FINAL CFR TRACKED VIDEO RENDERER
#
# COMPATIBILITY:
#   Kaggle FFmpeg 4.4.2
#
# IMPORTANT:
# ------------------------------------------------------------
# This cell DOES NOT:
#   - run YOLO
#   - run BoT-SORT
#   - recalculate tracking
#   - recalculate counting
#
# It ONLY renders the already-computed Phase 2 + Phase 3 data.
#
# OUTPUT:
#   - every source frame is rendered
#   - no frame dropping
#   - CFR output
#   - track-level class
#   - diagonal counting line
#   - cumulative vehicle count
#   - crossing events
# ============================================================

import cv2
import time
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path


print("=" * 70)
print("PHASE 3 — FINAL CFR TRACKED VIDEO")
print("=" * 70)


# ============================================================
# PATHS
# ============================================================

PHASE3_TRACKED_VIDEO_PATH = (
    PHASE3_DIR /
    "tracked_video_diagonal_counting_CFR.mp4"
)

FFMPEG_LOG_PATH = (
    PHASE3_DIR /
    "ffmpeg_render.log"
)


# Remove previous failed output
if PHASE3_TRACKED_VIDEO_PATH.exists():

    PHASE3_TRACKED_VIDEO_PATH.unlink()


if FFMPEG_LOG_PATH.exists():

    FFMPEG_LOG_PATH.unlink()


print(
    f"Output video : "
    f"{PHASE3_TRACKED_VIDEO_PATH}"
)

print(
    f"FFmpeg log   : "
    f"{FFMPEG_LOG_PATH}"
)


# ============================================================
# OPEN SOURCE VIDEO
# ============================================================

cap = cv2.VideoCapture(
    str(VIDEO_PATH)
)


if not cap.isOpened():

    raise RuntimeError(
        f"Cannot open video:\n"
        f"{VIDEO_PATH}"
    )


# ============================================================
# READ VIDEO METADATA
# ============================================================

FPS = cap.get(
    cv2.CAP_PROP_FPS
)

FRAME_WIDTH = int(
    cap.get(
        cv2.CAP_PROP_FRAME_WIDTH
    )
)

FRAME_HEIGHT = int(
    cap.get(
        cv2.CAP_PROP_FRAME_HEIGHT
    )
)

TOTAL_FRAMES = int(
    cap.get(
        cv2.CAP_PROP_FRAME_COUNT
    )
)


print("\nVIDEO")
print("-" * 70)

print(
    f"Resolution : "
    f"{FRAME_WIDTH} x {FRAME_HEIGHT}"
)

print(
    f"FPS        : "
    f"{FPS:.12f}"
)

print(
    f"Frames     : "
    f"{TOTAL_FRAMES:,}"
)


# ============================================================
# COUNTING LINE
#
# Diagonal line:
#
# Upper-right
#       \
#        \
#         \
#          \
#           Lower-left
# ============================================================

COUNT_LINE = {

    "orientation": "diagonal",

    "x1": int(
        FRAME_WIDTH * 0.95
    ),

    "y1": int(
        FRAME_HEIGHT * 0.20
    ),

    "x2": int(
        FRAME_WIDTH * 0.05
    ),

    "y2": int(
        FRAME_HEIGHT * 0.95
    ),
}


print("\nCOUNTING LINE")
print("-" * 70)

print(
    COUNT_LINE
)


# ============================================================
# PREPARE CUMULATIVE COUNT ARRAY
# ============================================================

if isinstance(
    cumulative_count_by_frame,
    pd.Series
):

    cumulative_count_array = (
        cumulative_count_by_frame
        .reset_index(drop=True)
        .to_numpy(
            dtype=np.int64
        )
    )

else:

    cumulative_count_array = np.asarray(
        cumulative_count_by_frame,
        dtype=np.int64
    )


# ============================================================
# HANDLE POSSIBLE 1-BASED COUNT ARRAY
# ============================================================

if len(
    cumulative_count_array
) == TOTAL_FRAMES + 1:

    cumulative_count_array = (
        cumulative_count_array[1:]
    )


# ============================================================
# COUNT ARRAY VALIDATION
# ============================================================

if len(
    cumulative_count_array
) != TOTAL_FRAMES:

    cap.release()

    raise ValueError(

        "Cumulative count length mismatch.\n\n"

        f"Video frames : "
        f"{TOTAL_FRAMES:,}\n"

        f"Count array  : "
        f"{len(cumulative_count_array):,}"
    )


if np.any(
    cumulative_count_array < 0
):

    cap.release()

    raise ValueError(
        "Negative cumulative count detected."
    )


if np.any(
    np.diff(
        cumulative_count_array
    ) < 0
):

    cap.release()

    raise ValueError(
        "Cumulative count is not monotonic."
    )


print("\nCOUNT DATA")
print("-" * 70)

print(
    f"Count array length : "
    f"{len(cumulative_count_array):,}"
)

print(
    f"Final vehicle count: "
    f"{int(cumulative_count_array[-1])}"
)


# ============================================================
# FFMPEG COMMAND
#
# IMPORTANT:
# ------------------------------------------------------------
# FFmpeg 4.4.2 does NOT support:
#
#     -fps_mode cfr
#
# Therefore we use:
#
#     -vsync cfr
#
# which is supported by FFmpeg 4.4.x.
# ============================================================

ffmpeg_cmd = [

    "ffmpeg",

    "-y",

    "-hide_banner",

    "-loglevel",
    "error",

    # --------------------------------------------------------
    # RAWVIDEO INPUT
    # --------------------------------------------------------

    "-f",
    "rawvideo",

    "-pixel_format",
    "bgr24",

    "-video_size",
    f"{FRAME_WIDTH}x{FRAME_HEIGHT}",

    "-framerate",
    f"{FPS:.12f}",

    "-i",
    "pipe:0",

    # --------------------------------------------------------
    # VIDEO ENCODER
    # --------------------------------------------------------

    "-an",

    "-c:v",
    "libx264",

    "-preset",
    "medium",

    "-crf",
    "18",

    "-pix_fmt",
    "yuv420p",

    # --------------------------------------------------------
    # CFR
    #
    # Compatible with FFmpeg 4.4.2
    # --------------------------------------------------------

    "-vsync",
    "cfr",

    # --------------------------------------------------------
    # MP4
    # --------------------------------------------------------

    "-movflags",
    "+faststart",

    str(
        PHASE3_TRACKED_VIDEO_PATH
    ),
]


print("\nFFMPEG COMMAND")
print("-" * 70)

print(
    " ".join(
        ffmpeg_cmd
    )
)


# ============================================================
# START FFMPEG
# ============================================================

process = subprocess.Popen(

    ffmpeg_cmd,

    stdin=subprocess.PIPE,

    stdout=subprocess.DEVNULL,

    stderr=subprocess.PIPE,

    bufsize=0,
)


frame_id = 0

start_time = time.time()

render_error = None


# ============================================================
# FRAME RENDER LOOP
# ============================================================

try:

    while True:

        success, frame = (
            cap.read()
        )


        if not success:

            break


        frame_id += 1


        # ====================================================
        # FRAME VALIDATION
        # ====================================================

        if frame.shape != (
            FRAME_HEIGHT,
            FRAME_WIDTH,
            3
        ):

            raise RuntimeError(

                "Unexpected frame shape.\n"

                f"Frame ID : {frame_id}\n"

                f"Expected : "
                f"({FRAME_HEIGHT}, "
                f"{FRAME_WIDTH}, 3)\n"

                f"Actual   : "
                f"{frame.shape}"
            )


        # ====================================================
        # DRAW COUNTING LINE
        # ====================================================

        frame = draw_counting_line(
            frame
        )


        # ====================================================
        # DRAW TRACKS
        #
        # IMPORTANT:
        #
        # track_class
        # = Phase 2 track-level class
        #
        # NEVER use raw class_name here.
        # ====================================================

        group = (
            frame_groups.get(
                frame_id
            )
        )


        if group is not None:

            for row in group.itertuples(
                index=False
            ):

                row_data = {

                    "x1": float(
                        row.x1
                    ),

                    "y1": float(
                        row.y1
                    ),

                    "x2": float(
                        row.x2
                    ),

                    "y2": float(
                        row.y2
                    ),

                    "track_id": int(
                        row.track_id
                    ),

                    "track_class": str(
                        row.track_class
                    ),

                    "track_class_ratio": float(
                        row.track_class_ratio
                    ),

                    "class_ambiguous": bool(
                        row.class_ambiguous
                    ),
                }


                frame = draw_track(
                    frame,
                    row_data
                )


        # ====================================================
        # CURRENT CUMULATIVE VEHICLE COUNT
        # ====================================================

        current_count = int(

            cumulative_count_array[
                frame_id - 1
            ]

        )


        # ====================================================
        # COUNT PANEL
        # ====================================================

        cv2.rectangle(

            frame,

            (
                15,
                10
            ),

            (
                370,
                82
            ),

            (0, 0, 0),

            -1
        )


        cv2.putText(

            frame,

            (
                f"VEHICLE COUNT: "
                f"{current_count}"
            ),

            (
                25,
                40
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.65,

            (255, 255, 255),

            2,

            cv2.LINE_AA
        )


        cv2.putText(

            frame,

            "MOTORCYCLE + RIDER = 1",

            (
                25,
                65
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.38,

            (200, 200, 200),

            1,

            cv2.LINE_AA
        )


        # ====================================================
        # CROSSING EVENTS
        #
        # These events have already been calculated
        # by Phase 3.
        # ====================================================

        events = (
            crossing_frame_groups.get(
                frame_id
            )
        )


        if events is not None:

            max_events = max(

                1,

                int(
                    (
                        FRAME_HEIGHT - 40
                    ) / 25
                )

            )


            for offset, event in enumerate(

                events.itertuples(
                    index=False
                )

            ):

                if offset >= max_events:

                    break


                event_text = (

                    f"COUNTED: "

                    f"{str(event.track_class).upper()} "

                    f"| ID {int(event.track_id)} "

                    f"| {event.direction}"

                )


                y = (

                    FRAME_HEIGHT
                    -
                    25
                    -
                    (
                        offset * 24
                    )

                )


                cv2.putText(

                    frame,

                    event_text,

                    (
                        20,
                        y
                    ),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.48,

                    (0, 255, 255),

                    2,

                    cv2.LINE_AA
                )


        # ====================================================
        # FRAME NUMBER
        # ====================================================

        cv2.putText(

            frame,

            (
                f"Frame: "
                f"{frame_id:,}"
            ),

            (
                FRAME_WIDTH - 180,
                30
            ),

            cv2.FONT_HERSHEY_SIMPLEX,

            0.45,

            (255, 255, 255),

            1,

            cv2.LINE_AA
        )


        # ====================================================
        # SEND EXACTLY ONE FRAME TO FFMPEG
        # ====================================================

        try:

            process.stdin.write(
                frame.tobytes()
            )

        except BrokenPipeError:

            render_error = (
                f"FFmpeg closed pipe "
                f"at frame {frame_id:,}."
            )

            break


finally:

    cap.release()


# ============================================================
# CLOSE PIPE
# ============================================================

try:

    process.stdin.close()

except Exception:

    pass


# ============================================================
# READ FFMPEG STDERR
# ============================================================

ffmpeg_stderr = (
    process.stderr
    .read()
    .decode(
        "utf-8",
        errors="replace"
    )
)


# ============================================================
# WAIT FOR PROCESS
# ============================================================

return_code = (
    process.wait()
)


elapsed = (
    time.time()
    -
    start_time
)


# ============================================================
# SAVE FFMPEG LOG
# ============================================================

FFMPEG_LOG_PATH.write_text(
    ffmpeg_stderr
)


# ============================================================
# HANDLE FFMPEG ERROR
# ============================================================

if return_code != 0:

    print("\n" + "=" * 70)
    print("FFMPEG FAILED")
    print("=" * 70)

    print(
        f"Return code : "
        f"{return_code}"
    )

    print(
        f"Last frame  : "
        f"{frame_id:,}"
    )

    print(
        "\nFFmpeg error:"
    )

    print(
        ffmpeg_stderr
    )

    raise RuntimeError(
        "FFmpeg rendering failed."
    )


if render_error is not None:

    raise RuntimeError(
        render_error
    )


# ============================================================
# FRAME COUNT VALIDATION
# ============================================================

if frame_id != TOTAL_FRAMES:

    raise RuntimeError(

        "Source video was not fully rendered.\n"

        f"Rendered frames : "
        f"{frame_id:,}\n"

        f"Expected frames : "
        f"{TOTAL_FRAMES:,}"
    )


# ============================================================
# OUTPUT VALIDATION
# ============================================================

if not (
    PHASE3_TRACKED_VIDEO_PATH.exists()
):

    raise RuntimeError(
        "FFmpeg finished but output "
        "video does not exist."
    )


output_size_mb = (

    PHASE3_TRACKED_VIDEO_PATH
    .stat()
    .st_size
    /
    (1024 ** 2)

)


if output_size_mb <= 0:

    raise RuntimeError(
        "Output video is empty."
    )


# ============================================================
# SUCCESS
# ============================================================

print("\n" + "=" * 70)
print("VIDEO GENERATION COMPLETE")
print("=" * 70)

print(
    f"Frames rendered : "
    f"{frame_id:,}"
)

print(
    f"Expected frames : "
    f"{TOTAL_FRAMES:,}"
)

print(
    f"Frame validation: "
    f"{frame_id == TOTAL_FRAMES}"
)

print(
    f"FPS             : "
    f"{FPS:.12f}"
)

print(
    f"Elapsed         : "
    f"{elapsed:.2f} sec"
)

print(
    f"Render FPS      : "
    f"{frame_id / elapsed:.2f}"
)

print(
    f"Final count     : "
    f"{int(cumulative_count_array[-1])}"
)

print(
    f"Output          : "
    f"{PHASE3_TRACKED_VIDEO_PATH}"
)

print(
    f"File size       : "
    f"{output_size_mb:.2f} MB"
)

PHASE 3 — FINAL CFR TRACKED VIDEO
Output video : /kaggle/working/traffic_counting/phase3/tracked_video_diagonal_counting_CFR.mp4
FFmpeg log   : /kaggle/working/traffic_counting/phase3/ffmpeg_render.log

VIDEO
----------------------------------------------------------------------
Resolution : 1280 x 720
FPS        : 29.815016602561
Frames     : 10,282

COUNTING LINE
----------------------------------------------------------------------
{'orientation': 'diagonal', 'x1': 1216, 'y1': 144, 'x2': 64, 'y2': 684}

COUNT DATA
----------------------------------------------------------------------
Count array length : 10,282
Final vehicle count: 75

FFMPEG COMMAND
----------------------------------------------------------------------
ffmpeg -y -hide_banner -loglevel error -f rawvideo -pixel_format bgr24 -video_size 1280x720 -framerate 29.815016602561 -i pipe:0 -an -c:v libx264 -preset medium -crf 18 -pix_fmt yuv420p -vsync cfr -movflags +faststart /kaggle/working/traffic_counting/phase3/tracked_v

In [33]:
# ============================================================
# PHASE 3 — FFmpeg ERROR DIAGNOSTIC
# ============================================================

from pathlib import Path

FFMPEG_LOG_PATH = (
    PHASE3_DIR /
    "ffmpeg_render.log"
)

print("=" * 70)
print("FFMPEG DIAGNOSTIC")
print("=" * 70)

print(
    "Log path:",
    FFMPEG_LOG_PATH
)

if not FFMPEG_LOG_PATH.exists():

    raise FileNotFoundError(
        f"FFmpeg log not found:\n"
        f"{FFMPEG_LOG_PATH}"
    )


log_text = FFMPEG_LOG_PATH.read_text(
    errors="replace"
)

print("\n" + "=" * 70)
print("LAST 100 LINES")
print("=" * 70)

lines = log_text.splitlines()

for line in lines[-100:]:
    print(line)

FFMPEG DIAGNOSTIC
Log path: /kaggle/working/traffic_counting/phase3/ffmpeg_render.log

LAST 100 LINES
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libss

In [36]:
# ============================================================
# PHASE 3 — SOURCE vs OUTPUT VIDEO VALIDATION
# ============================================================

def probe_video(path):

    cap = cv2.VideoCapture(
        str(path)
    )

    if not cap.isOpened():

        raise RuntimeError(
            f"Cannot open: {path}"
        )

    metadata = {

        "path": str(path),

        "width": int(
            cap.get(
                cv2.CAP_PROP_FRAME_WIDTH
            )
        ),

        "height": int(
            cap.get(
                cv2.CAP_PROP_FRAME_HEIGHT
            )
        ),

        "fps": cap.get(
            cv2.CAP_PROP_FPS
        ),

        "frame_count": int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        ),

    }

    if metadata["fps"] > 0:

        metadata["duration_sec"] = (
            metadata["frame_count"]
            /
            metadata["fps"]
        )

    else:

        metadata["duration_sec"] = None

    cap.release()

    return metadata


source_info = probe_video(
    VIDEO_PATH
)

output_info = probe_video(
    PHASE3_TRACKED_VIDEO_PATH
)


validation = pd.DataFrame(

    [
        source_info,
        output_info
    ],

    index=[
        "SOURCE",
        "OUTPUT"
    ]

)


display(
    validation
)

,path,width,height,fps,frame_count,duration_sec
SOURCE,/kaggle/input/datasets/chrisbiran/traffic-trac...,1280,720,29.815017,10282,344.859778
OUTPUT,/kaggle/working/traffic_counting/phase3/tracke...,1280,720,29.815017,10282,344.859778


In [38]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("=" * 70)
print("PHASE 3 VIDEO VALIDATION")
print("=" * 70)


checks = {

    "width":
        source_info["width"]
        ==
        output_info["width"],

    "height":
        source_info["height"]
        ==
        output_info["height"],

    "frame_count":
        source_info["frame_count"]
        ==
        output_info["frame_count"],

    "fps":
        abs(
            source_info["fps"]
            -
            output_info["fps"]
        ) < 0.01,

    "duration":
        abs(
            source_info["duration_sec"]
            -
            output_info["duration_sec"]
        ) < 0.10,
}


for name, passed in checks.items():

    print(
        f"{name:15s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


print("\nOverall:")

if all(checks.values()):

    print(
        "PASS — SOURCE AND OUTPUT VIDEO "
        "TIMELINE ARE CONSISTENT."
    )

else:

    print(
        "WARNING — VIDEO METADATA "
        "DIFFERS."
    )

PHASE 3 VIDEO VALIDATION
width          : PASS
height         : PASS
frame_count    : PASS
fps            : PASS
duration       : PASS

Overall:
PASS — SOURCE AND OUTPUT VIDEO TIMELINE ARE CONSISTENT.
